# Paso 12: Tablas 5 y 6 con validación por paciente

Cierra las dos notas pendientes del artículo rehaciendo, bajo el **mismo protocolo del paso 11**, las dos tablas que todavía provienen del protocolo preliminar:

- **Tabla 5.** La regla determinista de 50–100 BPM frente a los modelos agregados por segmentos de 10 s.
- **Tabla 6.** El efecto de la representación de entrada: XGBoost y la red convolucional, cada uno con 26 puntos y con las 252 muestras.

Reutiliza exactamente la misma partición por paciente del paso 11 (los mismos cuatro registros de DS1 apartados) para que todas las tablas del artículo queden bajo un único protocolo.

Se omiten la red recurrente, el ajuste de umbral, las cinco semillas y el remuestreo, porque ninguna de las dos tablas los necesita. Por eso baja de hora y cuarto a unos **25 minutos**.

**Antes de ejecutar:** activa la **T4 GPU** en Entorno de ejecución → Cambiar tipo de entorno de ejecución.

## 1. Instalar y descargar

In [ ]:
!pip install wfdb xgboost -q
import wfdb, time
import numpy as np, pandas as pd
from scipy.signal import lfilter
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, recall_score, precision_score

wfdb.dl_database('mitdb', dl_dir='mitdb')
print('Descarga lista')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.3 MB/s eta 0:00:00
Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating record list for: 108
Generating record list for: 109
Generating record list for: 111
Generating record list for: 112
Generating record list for: 113
Generating record list for: 114
Generating record list for: 115
Generating record list for: 116
Generating record list for: 117
Generating record list for: 118
Generating record list for: 119
Generating record list for: 121
Generating record list for: 122
Generating record list for: 123
Generating record list for: 124
Generating record list for: 200
Generating record list for: 201
Generating record list for: 202
Generating record list for: 203
Generating record list for: 205
Generating record list f

## 2. Extraer los latidos

Se guarda, además de la forma y los descriptores RR, la **posición del pico R** y el **registro** de cada latido: la primera hace falta para construir los segmentos de 10 s, el segundo para apartar registros completos.

In [ ]:
DS1 = [101,106,108,109,112,114,115,116,118,119,122,124,201,203,205,207,208,209,215,220,223,230]
DS2 = [100,103,105,111,113,117,121,123,200,202,210,212,213,214,219,221,222,228,231,232,233,234]
AAMI = {'N':'N','L':'N','R':'N','e':'N','j':'N','A':'S','a':'S','J':'S','S':'S','V':'V','E':'V','F':'F'}
CODIGOS_LATIDO = set('NLRejAaJSVEF/fQ')
CLASES = ['N','S','V','F']
FS, ANTES, DESPUES = 360, 90, 162
VENTANA_SEG = 10
MUESTRAS_SEG = FS * VENTANA_SEG

def filtro_pasa_altas(x, fs=FS, fc=1.0):
    alpha = np.exp(-2*np.pi*fc/fs)
    return x - lfilter([1-alpha], [1, -alpha], x)

def extraer_latidos(num):
    ruta = f'mitdb/{num}'
    rec = wfdb.rdrecord(ruta); ann = wfdb.rdann(ruta, 'atr')
    canal = rec.sig_name.index('MLII') if 'MLII' in rec.sig_name else 0
    x = filtro_pasa_altas(rec.p_signal[:, canal]); x = (x - x.mean())/x.std()
    pos = np.array([p for p, s in zip(ann.sample, ann.symbol) if s in CODIGOS_LATIDO])
    sim = [s for s in ann.symbol if s in CODIGOS_LATIDO]
    rr = np.diff(pos)/FS; rr_medio = rr.mean()
    V, R, Y, P, G = [], [], [], [], []
    for i in range(1, len(pos)-1):
        if sim[i] not in AAMI: continue
        r = pos[i]
        if r-ANTES < 0 or r+DESPUES > len(x): continue
        V.append(x[r-ANTES : r+DESPUES])
        R.append([rr[i-1], rr[i], rr[max(0,i-10):i].mean(), rr[i-1]/rr_medio, rr[i]/rr_medio])
        Y.append(AAMI[sim[i]]); P.append(int(r)); G.append(int(num))
    return V, R, Y, P, G

def construir(lista):
    V, R, Y, P, G = [], [], [], [], []
    for n in lista:
        v, r, y, p, g = extraer_latidos(n)
        V += v; R += r; Y += y; P += p; G += g
    return (np.array(V, dtype='float32'), np.array(R, dtype='float32'),
            np.array(Y), np.array(P), np.array(G))

V_ent, R_ent, y_ent, _,       reg_ent = construir(DS1)
V_pru, R_pru, y_pru, pos_pru, reg_pru = construir(DS2)
a_numero = {c:i for i,c in enumerate(CLASES)}
y_ent_num = np.array([a_numero[c] for c in y_ent])
y_pru_num = np.array([a_numero[c] for c in y_pru])
print('DS1:', len(y_ent_num), 'latidos |', len(set(reg_ent)), 'registros')
print('DS2:', len(y_pru_num), 'latidos |', len(set(reg_pru)), 'registros')

DS1: 50969 latidos | 22 registros
DS2: 49661 latidos | 22 registros


## 3. La misma partición por paciente del paso 11

Este bloque es idéntico al del paso 11, con la misma semilla, de modo que aparta **los mismos cuatro registros**. Compruébalo contra la salida de aquel cuaderno antes de seguir.

In [ ]:
N_VAL = 4
MIN_POR_CLASE = 15
rng_sel = np.random.default_rng(42)
registros_ds1 = sorted(set(reg_ent))

eleccion, mejor = None, -1
for _ in range(500):
    cand = sorted(rng_sel.choice(registros_ds1, N_VAL, replace=False).tolist())
    m = np.isin(reg_ent, cand)
    cuenta = [int((y_ent_num[m] == i).sum()) for i in range(4)]
    if min(cuenta) >= MIN_POR_CLASE:
        eleccion = cand; break
    if min(cuenta) > mejor:
        mejor, eleccion = min(cuenta), cand

val_reg = eleccion
es_val = np.isin(reg_ent, val_reg)
V_tr, R_tr, y_tr = V_ent[~es_val], R_ent[~es_val], y_ent_num[~es_val]
V_val, R_val, y_val = V_ent[es_val], R_ent[es_val], y_ent_num[es_val]

print('Registros de validacion:', val_reg)
print('>>> Deben coincidir con los que imprimio el paso 11 <<<')
print()
print('Latidos entrenamiento:', len(y_tr), '| validacion:', len(y_val))

Registros de validacion: [112, 124, 205, 223]
>>> Deben coincidir con los que imprimio el paso 11 <<<

Latidos entrenamiento: 41558 | validacion: 9411


## 4. Representaciones y pesos de clase

Las cuatro combinaciones de la Tabla 6 comparten los cinco descriptores RR; lo único que cambia es la morfología: 26 puntos submuestreados o las 252 muestras completas.

In [ ]:
X_tr_26,  X_pru_26  = np.hstack([R_tr, V_tr[:, ::10]]), np.hstack([R_pru, V_pru[:, ::10]])
X_tr_252, X_pru_252 = np.hstack([R_tr, V_tr]),         np.hstack([R_pru, V_pru])

pesos_v = compute_class_weight('balanced', classes=np.arange(4), y=y_tr)
peso_muestra = pesos_v[y_tr]
pesos_clase = {i: w for i, w in enumerate(pesos_v)}

predicciones = {}

def registrar(nombre, pred):
    predicciones[nombre] = pred
    se = recall_score(y_pru_num, pred, labels=range(4), average=None, zero_division=0)
    f1 = f1_score(y_pru_num, pred, labels=range(4), average=None, zero_division=0)
    print(f'  {nombre:26s} F1(4)={f1.mean():.3f}  F1(3)={f1[:3].mean():.3f}  Se_S={se[1]:.3f}')
    return {'F1_macro_4': f1.mean(), 'F1_macro_3': f1[:3].mean(), 'Se_S': se[1]}

resultados_repr = {}

## 5. Modelos de árboles

In [ ]:
print('Random Forest (26 puntos)')
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=0)
rf.fit(X_tr_26, y_tr)
registrar('Random Forest', rf.predict(X_pru_26))
print('  nodos:', sum(t.tree_.node_count for t in rf.estimators_))

from xgboost import XGBClassifier
print()
print('XGBoost')
for etiqueta, Xe, Xp in [('XGBoost (26 puntos)',  X_tr_26,  X_pru_26),
                         ('XGBoost (252 muestras)', X_tr_252, X_pru_252)]:
    m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method='hist',
                      objective='multi:softprob', num_class=4, random_state=0, n_jobs=-1)
    m.fit(Xe, y_tr, sample_weight=peso_muestra)
    resultados_repr[etiqueta] = registrar(etiqueta, m.predict(Xp))
    if etiqueta.startswith('XGBoost (26'):
        print('  nodos:', len(m.get_booster().trees_to_dataframe()))

Random Forest (26 puntos)
  Random Forest              F1(4)=0.452  F1(3)=0.601  Se_S=0.020
  nodos: 209704

XGBoost
  XGBoost (26 puntos)        F1(4)=0.503  F1(3)=0.655  Se_S=0.124
  nodos: 55146
  XGBoost (252 muestras)     F1(4)=0.521  F1(3)=0.669  Se_S=0.124


## 6. Red convolucional, con las 252 muestras y con 26 puntos

Misma arquitectura e hiperparámetros del paso 11. La detención temprana usa los cuatro registros apartados.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def armar_cnn(largo):
    ef = keras.Input(shape=(largo, 1))
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(ef)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 3, activation='relu', padding='same')(x)
    x = layers.GlobalAveragePooling1D()(x)
    er = keras.Input(shape=(5,))
    z = layers.concatenate([x, er])
    z = layers.Dense(64, activation='relu')(z)
    z = layers.Dropout(0.3)(z)
    s = layers.Dense(4, activation='softmax')(z)
    m = keras.Model([ef, er], s)
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005, clipnorm=1.0),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def entrenar_cnn(Vtr, Vval, Vpru, semilla=0):
    keras.utils.set_random_seed(semilla)
    m = armar_cnn(Vtr.shape[1])
    m.fit([Vtr[..., None], R_tr], y_tr,
          validation_data=([Vval[..., None], R_val], y_val),
          epochs=40, batch_size=256, class_weight=pesos_clase,
          callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=8,
                                                   restore_best_weights=True),
                     keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4)],
          verbose=0)
    return m, m.predict([Vpru[..., None], R_pru], verbose=0).argmax(1)

print('Red convolucional')
cnn252, pred252 = entrenar_cnn(V_tr, V_val, V_pru)
resultados_repr['CNN 1D (252 muestras)'] = registrar('CNN 1D (252 muestras)', pred252)
predicciones['CNN 1D'] = pred252          # la que usa la Tabla 5
print('  parametros:', cnn252.count_params())

cnn26, pred26 = entrenar_cnn(V_tr[:, ::10], V_val[:, ::10], V_pru[:, ::10])
resultados_repr['CNN 1D (26 puntos)'] = registrar('CNN 1D (26 puntos)', pred26)

Red convolucional
  CNN 1D (252 muestras)      F1(4)=0.558  F1(3)=0.743  Se_S=0.803
  parametros: 27652
  CNN 1D (26 puntos)         F1(4)=0.413  F1(3)=0.549  Se_S=0.149


## 7. Tabla 6: efecto de la representación

In [ ]:
orden6 = ['XGBoost (26 puntos)', 'XGBoost (252 muestras)',
          'CNN 1D (26 puntos)', 'CNN 1D (252 muestras)']
tabla6 = pd.DataFrame(resultados_repr).T.reindex(orden6).round(3)
print('TABLA 6 — EFECTO DE LA REPRESENTACION, CON VALIDACION POR PACIENTE')
print(tabla6.to_string())
tabla6.to_csv('a4_tabla6_representacion.csv')

TABLA 6 — EFECTO DE LA REPRESENTACION, CON VALIDACION POR PACIENTE
                        F1_macro_4  F1_macro_3   Se_S
XGBoost (26 puntos)          0.503       0.655  0.124
XGBoost (252 muestras)       0.521       0.669  0.124
CNN 1D (26 puntos)           0.413       0.549  0.149
CNN 1D (252 muestras)        0.558       0.743  0.803


## 8. Segmentos de 10 s

Un segmento es anormal cuando contiene al menos un latido distinto de N. Se descartan los segmentos con menos de tres latidos, donde no se puede estimar el ritmo.

In [ ]:
segmentos = []
for g in sorted(set(reg_pru)):
    m = reg_pru == g
    p, y = pos_pru[m], y_pru_num[m]
    idx_global = np.where(m)[0]
    bloque = p // MUESTRAS_SEG
    for b in sorted(set(bloque)):
        sel = bloque == b
        if sel.sum() < 3: continue
        rr = np.diff(p[sel]) / FS
        segmentos.append({'registro': g,
                          'bpm_rr': 60.0 / rr.mean(),
                          'bpm_conteo': sel.sum() * (60.0 / VENTANA_SEG),
                          'anormal_real': bool((y[sel] != 0).any()),
                          'indices': idx_global[sel]})
seg = pd.DataFrame(segmentos)
print('Segmentos de', VENTANA_SEG, 's:', len(seg))
print('Con al menos un latido anormal:', int(seg['anormal_real'].sum()),
      f"({100*seg['anormal_real'].mean():.1f} %)")
print()
print('>>> Compara estos dos numeros con los del paso 10: deben ser identicos,')
print('    porque los segmentos no dependen del modelo ni de la particion. <<<')

Segmentos de 10 s: 3982
Con al menos un latido anormal: 1472 (37.0 %)

>>> Compara estos dos numeros con los del paso 10: deben ser identicos,
    porque los segmentos no dependen del modelo ni de la particion. <<<


## 9. Tabla 5: la regla determinista frente a los modelos

In [ ]:
def binarias(real, pred):
    vp = int((real & pred).sum()); fp = int((~real & pred).sum())
    fn = int((real & ~pred).sum()); vn = int((~real & ~pred).sum())
    se = vp / (vp + fn) if vp + fn else 0.0
    pp = vp / (vp + fp) if vp + fp else 0.0
    f1 = 2*se*pp/(se+pp) if se + pp else 0.0
    return {'Se': se, '+P': pp, 'F1': f1, 'Exactitud': (vp+vn)/len(real),
            'VP': vp, 'FP': fp, 'FN': fn, 'VN': vn}

real = seg['anormal_real'].values
filas5 = {}

for etiqueta, col in [('Regla determinista, ritmo por intervalos RR', 'bpm_rr'),
                      ('Regla determinista, ritmo por conteo de latidos', 'bpm_conteo')]:
    bpm = seg[col].values
    filas5[etiqueta] = binarias(real, (bpm < 50) | (bpm > 100))

for nombre in ['Random Forest', 'XGBoost (26 puntos)', 'CNN 1D']:
    pred = predicciones[nombre]
    pa = np.array([bool((pred[s] != 0).any()) for s in seg['indices']])
    filas5[nombre] = binarias(real, pa)

tabla5 = pd.DataFrame(filas5).T
print('TABLA 5 — DETECCION DE SEGMENTOS CON LATIDOS ANORMALES (DS2)')
print(tabla5.round(3).to_string())
tabla5.to_csv('a4_tabla5_regla.csv')

TABLA 5 — DETECCION DE SEGMENTOS CON LATIDOS ANORMALES (DS2)
                                                    Se     +P     F1  Exactitud      VP     FP      FN      VN
Regla determinista, ritmo por intervalos RR      0.228  0.531  0.319      0.640   336.0  297.0  1136.0  2213.0
Regla determinista, ritmo por conteo de latidos  0.232  0.453  0.307      0.613   341.0  412.0  1131.0  2098.0
Random Forest                                    0.753  0.860  0.803      0.864  1109.0  180.0   363.0  2330.0
XGBoost (26 puntos)                              0.935  0.782  0.852      0.880  1376.0  383.0    96.0  2127.0
CNN 1D                                           0.977  0.661  0.788      0.806  1438.0  738.0    34.0  1772.0


## 10. Descargar

In [ ]:
from google.colab import files
for a in ['a4_tabla5_regla.csv', 'a4_tabla6_representacion.csv']:
    files.download(a)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>